In [1]:
import pandas as pd
import numpy as np
from coffea import util
import itertools
import os, sys
import matplotlib.pyplot as plt
import mplhep as hep
import uproot
import hist
from matplotlib.collections import PatchCollection
from matplotlib.patches import Rectangle
hep.style.use("CMS")

sys.path.append('../python/')
from functions import loadCoffeaFile, getLabelMap, getCoffeaFilenames, plotBackgroundEstimate, getHist


In [2]:
# IOVs = ['2016APV', '2016', '2016all', '2017', '2018', 'Full']
IOVs = ['2016all']
useOldHTcut = True

lumi = {
    "2016APV": 19800.,
    "2016": 16120., #35920 - 19800
    "2016all": 35920,
    "2017": 41530./10.,
    "2018": 59800./10., #59740./10., #Blinding
    "Full": 46053. # 137190. Blinded
}

t_BR = 0.6741
ttbar_BR = 0.4544 #PDG 2019
ttbar_xs1 = 831.76 * (0.09210) #pb For ttbar mass from 700 to 1000
ttbar_xs2 = 831.76 * (0.02474) #pb For ttbar mass from 1000 to Inf
toptag_sf = 1.0 #0.9 # Now included as a weight in processing
toptag_kf = 1.0 #0.7
qcd_xs = 1370000000.0 #pb From https://cms-gen-dev.cern.ch/xsdb

systematics = [
        'jes',
        'jer',
        'pileup',
        'pdf',
        'q2',
        'btag',
        'toptagsf',
        'toptagxs',
        'lumi',
        'prefiring'
    ]

oldHTstr = ''
if useOldHTcut:
    oldHTstr = '_oldHTcut'
    
# systematics = ['btag']

In [14]:
# load histograms and get scale factors
coffeaFiles = getCoffeaFilenames(False, useOldHTcut)

LoadedFiles = {
    'TTbar': {
        'unweighted': {'2016APV':[], '2016':[]},
        'weighted': {'2016APV':[], '2016':[]}
    },
    'JetHT': {
        'unweighted': {'2016APV':[], '2016':[]},
        'weighted': {'2016APV':[], '2016':[]}
    },
    'QCD': {
        'unweighted': {'2016APV':[], '2016':[]},
        'weighted': {'2016APV':[], '2016':[]}
    },
    'RSGluon':{
        'unweighted':{
            '2016APV': {'1000':[], '1500':[], '2000':[], '2500':[], '3000':[], '3500':[], '4000':[], '4500':[], '5000':[]},
            '2016': {'1000':[], '1500':[], '2000':[], '2500':[], '3000':[], '3500':[], '4000':[], '4500':[], '5000':[]}
        }
    },
    'ZPrime1':{
        'unweighted':{
            '2016APV': {'1000':[], '1200':[], '1400':[], '1600':[], '1800':[], '2000':[], '2500':[], '3000':[], '3500':[], '4000':[], '4500':[]},
            '2016': {'1000':[], '1200':[], '1400':[], '1600':[], '1800':[], '2000':[], '2500':[], '3000':[], '3500':[], '4000':[], '4500':[]}
        }
    },
    'ZPrime10':{
        'unweighted':{
            '2016APV': {'1000':[], '1200':[], '1400':[], '1600':[], '1800':[], '2000':[], '2500':[], '3000':[], '3500':[], '4000':[], '4500':[], '5000':[]},
            '2016': {'1000':[], '1200':[], '1400':[], '1600':[], '1800':[], '2000':[], '2500':[], '3000':[], '3500':[], '4000':[], '4500':[], '5000':[]}
        }
    },
    'ZPrime30':{
        'unweighted':{
            '2016APV': {'1000':[], '1200':[], '1400':[], '1600':[], '1800':[], '2000':[], '2500':[], '3000':[], '3500':[], '4000':[], '4500':[], '5000':[]},
            '2016': {'1000':[], '1200':[], '1400':[], '1600':[], '1800':[], '2000':[], '2500':[], '3000':[], '3500':[], '4000':[], '4500':[], '5000':[]}
        }
    },
    'ZPrimeDM':{
        'unweighted':{
            '2016APV': {'1000':[], '1500':[], '2000':[], '2500':[], '3000':[], '3500':[], '4000':[], '4500':[], '5000':[]},
            '2016': {'1000':[], '1500':[], '2000':[], '2500':[], '3000':[], '3500':[], '4000':[], '4500':[], '5000':[]}
        }
    }
}

for ds in ['TTbar', 'JetHT', 'QCD']:
    for bkgest_str in ['unweighted', 'weighted']:
        for year in ['2016APV', '2016']:
            for key, file in coffeaFiles[ds][bkgest_str][year].items():
                LoadedFiles[ds][bkgest_str][year].append(util.load(file))
                print(file + ' loaded')

bkgest_str = 'unweighted'
for year in ['2016APV', '2016']:
    for ds in ['RSGluon', 'ZPrime1', 'ZPrime10', 'ZPrime30', 'ZPrimeDM']:
        for key, file in coffeaFiles[ds][bkgest_str][year].items():
            LoadedFiles[ds][bkgest_str][year][key].append(util.load(file))
            print(file + ' loaded') # for masspoint ' + key)

/srv/outputs/TTbar_2016APV_700to1000_oldHTcut_oldBTag.coffea loaded
/srv/outputs/TTbar_2016APV_1000toInf_oldHTcut_oldBTag.coffea loaded
/srv/outputs/TTbar_2016_700to1000_oldHTcut_oldBTag.coffea loaded
/srv/outputs/TTbar_2016_1000toInf_oldHTcut_oldBTag.coffea loaded
/srv/outputs/TTbar_2016APV_700to1000_bkgest_oldHTcut_oldBTag.coffea loaded
/srv/outputs/TTbar_2016APV_1000toInf_bkgest_oldHTcut_oldBTag.coffea loaded
/srv/outputs/TTbar_2016_700to1000_bkgest_oldHTcut_oldBTag.coffea loaded
/srv/outputs/TTbar_2016_1000toInf_bkgest_oldHTcut_oldBTag.coffea loaded
/srv/outputs/JetHT_2016APVB_noSyst_oldHTcut_oldBTag.coffea loaded
/srv/outputs/JetHT_2016APVC_noSyst_oldHTcut_oldBTag.coffea loaded
/srv/outputs/JetHT_2016APVD_noSyst_oldHTcut_oldBTag.coffea loaded
/srv/outputs/JetHT_2016APVE_noSyst_oldHTcut_oldBTag.coffea loaded
/srv/outputs/JetHT_2016APVF_noSyst_oldHTcut_oldBTag.coffea loaded
/srv/outputs/JetHT_2016F_noSyst_oldHTcut_oldBTag.coffea loaded
/srv/outputs/JetHT_2016G_noSyst_oldHTcut_oldBTa

In [15]:
def make_error_boxes(ax, xdata, ydata, xerror, yerror, facecolor='none',
                     edgecolor='none', alpha=0.5):
    
    # Loop over data points; create box from errors at each point
    errorboxes = [Rectangle((x - xe, y - ye), xe.sum(), ye.sum()) for x, y, xe, ye in zip(xdata, ydata, xerror.T, yerror.T)]

    # Create patch collection with specified colour/alpha
    pc = PatchCollection(errorboxes, facecolor=facecolor, alpha=alpha,
                         edgecolor=edgecolor)

    # Add collection to axes
    ax.add_collection(pc)

    # Plot errorbars
    artists = ax.errorbar(xdata, ydata, xerr=xerror, yerr=yerror,
                          fmt='none', ecolor='k', barsabove=True)

    return artists

def getHist(hname, ds, bkgest, year, sum_axes=[], integrate_axes={}, masspoint=''):
    
    ######################################################################################
    # hname = histogram name (example: 'ttbarmass')                                      #
    # ds = dataset name (example: 'JetHT')                                               #
    # bkgest = boolean, True if bkg estimate applied                                     #
    # year = '2016APV' or '2016' or '2017' or '2018'                                     #
    # sum_axes = names of axes to sum over for scikit-hep/hist histogram                 #
    # integrate_axes = range to integrate over axis (example: {'anacat': [0,1,2,3,4,5]}) #
    ######################################################################################    

    
    # load histograms and get scale factors
    coffeaFiles = getCoffeaFilenames(False, useOldHTcut)
    
    cfiles = []
    sf = []
    bkgest_str = np.where([bkgest], 'weighted', 'unweighted')[0]
    # print(coffeaFiles[ds][bkgest_str][year])
    
     
    for key, file in coffeaFiles[ds][bkgest_str][year].items():
        if masspoint != '':
            if masspoint in key:
                loaded_file = LoadedFiles[ds][bkgest_str][year][masspoint]
                sum_axes_dict = {ax:sum for ax in sum_axes}
                histo = loaded_file[hname][integrate_axes][sum_axes_dict]
                histo = histo * (lumi[IOV] * 1.0 / loaded_file['cutflow']['sumw'])
                return histo

        loaded_file = LoadedFiles[ds][bkgest_str][year]
        cfiles.append(loaded_file)


        if 'TTbar' in ds and '700to1000' in key:
            sf.append(lumi[year] * ttbar_xs1 / (loaded_file['cutflow']['sumw']))
        elif 'TTbar' in ds and '1000toInf' in key:
            sf.append(lumi[year] * ttbar_xs2 / (loaded_file['cutflow']['sumw']))   
        elif 'QCD' in ds:
            sf.append(lumi[year] * qcd_xs / (loaded_file['cutflow']['sumw']))
        else:
            sf.append(1.)
        
    # sum or integrate axes for all hists from dataset eras or pt bins
    sum_axes_dict = {ax:sum for ax in sum_axes}
    
    hists = []
    
    for cfile in cfiles:
        hists.append(cfile[hname][integrate_axes][sum_axes_dict])    
    
    # sum all hists from dataset eras or pt bins
    
    histo = hists[0]*sf[0]
    if 'QCD' in ds:
        histo = histo / np.sum(histo.values())
    
    if (len(hists) > 1) and ('QCD' in ds):
        for i in range(len(hists) - 1) : 
            histNew = hists[i+1]*sf[i+1]
            histo = histo + histNew / np.sum(histNew.values())
    elif (len(hists) > 1) and ('QCD' not in ds):
        for i in range(len(hists) - 1): 
            histo = histo + hists[i+1]*sf[i+1]
        
            
    return histo

def getHistNoMassMod(hname, ds, year, sum_axes=[], integrate_axes={}):
    
    ######################################################################################
    # hname = histogram name (example: 'ttbarmass')                                      #
    # ds = dataset name (example: 'JetHT')                                               #
    # bkgest = boolean, True if bkg estimate applied                                     #
    # year = '2016APV' or '2016' or '2017' or '2018'                                     #
    # sum_axes = names of axes to sum over for scikit-hep/hist histogram                 #
    # integrate_axes = range to integrate over axis (example: {'anacat': [0,1,2,3,4,5]}) #
    ######################################################################################    

    
    # load histograms and get scale factors
    coffeaFiles = getCoffeaFilenames(False, useOldHTcut)
    
    cfiles = []
    sf = []
    noMassMod_str = 'noMassMod'
    
    for key, file in coffeaFiles[ds][noMassMod_str][year].items():
            
        loaded_file = util.load(file)
        cfiles.append(loaded_file)
        
        
        if 'TTbar' in ds and '700to1000' in key:
            sf.append(lumi[year] * ttbar_xs1 * toptag_sf**2 * toptag_kf / loaded_file['cutflow']['sumw'])
        elif 'TTbar' in ds and '1000toInf' in key:
            sf.append(lumi[year] * ttbar_xs2 * toptag_sf**2 * toptag_kf / loaded_file['cutflow']['sumw'])  
        else:
            sf.append(1.)
        
    # sum or integrate axes for all hists from dataset eras or pt bins
    sum_axes_dict = {ax:sum for ax in sum_axes}
    
    hists = []
    for cfile in cfiles:
        hists.append(cfile[hname][integrate_axes][sum_axes_dict])    
    
    # sum all hists from dataset eras or pt bins
    histo = hists[0]*sf[0]
    if len(hists) > 1:
        for i in range(len(hists) - 1): 
            histo = histo + hists[i+1]*sf[i+1]
            
            
    return histo

In [16]:
#def All2016Hists(Variable, category='', systematic):

Variable = 'ttbarmass_fine'
category = ''
systematic = 'nominal'
    
if category == '':
    signal_cats = [ i for label, i in label_to_int_dict.items() if '2t' in label]
    pretag_cats = [ i for label, i in label_to_int_dict.items() if 'pret' in label]
    anti_cats   = [ i for label, i in label_to_int_dict.items() if 'at' in label]

    httbar_apv = getHist(Variable, 'TTbar', False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':systematic})
    hcontam_apv = getHist(Variable, 'TTbar', True, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':systematic})
    hQCDsignal_apv = getHist(Variable, 'QCD', False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':systematic})
    hQCDbkgest_apv = getHist(Variable, 'QCD', True, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':systematic})
    hcontam_noMM_apv = getHistNoMassMod(Variable, 'TTbar', '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':systematic})
    hntmj_apv = getHist(Variable, 'JetHT', True, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':systematic})
    hntmj_noMM_apv = getHistNoMassMod(Variable, 'JetHT', '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':systematic})
    hdata_apv = getHist(Variable, 'JetHT', False, '2016APV',  sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':systematic})
    hpretag_apv = getHist(Variable, 'JetHT', False, '2016APV',sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':systematic}) 
    hantitag_data_apv = getHist(Variable, 'JetHT', False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':anti_cats, 'systematic':systematic})
    hantitag_ttbar_apv = getHist(Variable, 'TTbar', False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':anti_cats, 'systematic':systematic}) 

    httbar_noapv = getHist(Variable, 'TTbar', False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':systematic})
    hcontam_noapv = getHist(Variable, 'TTbar', True, '2016', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':systematic})
    hQCDsignal_noapv = getHist(Variable, 'QCD', False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':systematic})
    hQCDbkgest_noapv = getHist(Variable, 'QCD', True, '2016', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':systematic})
    hcontam_noMM_noapv = getHistNoMassMod(Variable, 'TTbar', '2016', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':systematic})
    hntmj_noapv = getHist(Variable, 'JetHT', True, '2016', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':systematic})
    hntmj_noMM_noapv = getHistNoMassMod(Variable, 'JetHT', '2016', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':systematic})
    hdata_noapv = getHist(Variable, 'JetHT', False, '2016',  sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':systematic})
    hpretag_noapv = getHist(Variable, 'JetHT', False, '2016',sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':systematic}) 
    hantitag_data_noapv = getHist(Variable, 'JetHT', False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':anti_cats, 'systematic':systematic})
    hantitag_ttbar_noapv = getHist(Variable, 'TTbar', False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':anti_cats, 'systematic':systematic}) 



else:
    signal_cats = label_to_int_dict['2t'+category]
    pretag_cats = label_to_int_dict['pret'+category]
    anti_cats   = label_to_int_dict['at'+category]

    httbar_apv = getHist(Variable, 'TTbar', False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':systematic})
    hcontam_apv = getHist(Variable, 'TTbar', True, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':systematic})
    hQCDsignal_apv = getHist(Variable, 'QCD', False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':systematic})
    hQCDbkgest_apv = getHist(Variable, 'QCD', True, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':systematic})
    hcontam_noMM_apv = getHistNoMassMod(Variable, 'TTbar', '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':systematic})
    hntmj_apv = getHist(Variable, 'JetHT', True, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':systematic})
    hntmj_noMM_apv = getHistNoMassMod(Variable, 'JetHT', '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':systematic})
    hdata_apv = getHist(Variable, 'JetHT', False, '2016APV',  sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':systematic})
    hpretag_apv = getHist(Variable, 'JetHT', False, '2016APV',sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':systematic}) 
    hantitag_data_apv = getHist(Variable, 'JetHT', False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':anti_cats, 'systematic':systematic})
    hantitag_ttbar_apv = getHist(Variable, 'TTbar', False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':anti_cats, 'systematic':systematic}) 

    httbar_noapv = getHist(Variable, 'TTbar', False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':systematic})
    hcontam_noapv = getHist(Variable, 'TTbar', True, '2016', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':systematic})
    hQCDsignal_noapv = getHist(Variable, 'QCD', False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':systematic})
    hQCDbkgest_noapv = getHist(Variable, 'QCD', True, '2016', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':systematic})
    hcontam_noMM_noapv = getHistNoMassMod(Variable, 'TTbar', '2016', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':systematic})
    hntmj_noapv = getHist(Variable, 'JetHT', True, '2016', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':systematic})
    hntmj_noMM_noapv = getHistNoMassMod(Variable, 'JetHT', '2016', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':systematic})
    hdata_noapv = getHist(Variable, 'JetHT', False, '2016',  sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':systematic})
    hpretag_noapv = getHist(Variable, 'JetHT', False, '2016',sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':systematic}) 
    hantitag_data_noapv = getHist(Variable, 'JetHT', False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':anti_cats, 'systematic':systematic})
    hantitag_ttbar_noapv = getHist(Variable, 'TTbar', False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':anti_cats, 'systematic':systematic}) 

httbar = httbar_apv + httbar_noapv
hcontam = hcontam_apv + hcontam_noapv
hQCDsignal = hQCDsignal_apv + hQCDsignal_noapv
hQCDbkgest = hQCDbkgest_apv + hQCDbkgest_noapv
hcontam_noMM = hcontam_noMM_apv + hcontam_noMM_noapv
hntmj = hntmj_apv + hntmj_noapv
hntmj_noMM = hntmj_noMM_apv + hntmj_noMM_noapv
hdata = hdata_apv + hdata_noapv
hpretag = hpretag_apv + hpretag_noapv
hantitag_data = hantitag_data_apv + hantitag_data_noapv
hantitag_ttbar = hantitag_ttbar_apv + hantitag_ttbar_noapv

hntmj_fixed = hntmj + -1*hcontam
hntmj_fixed_noMM = hntmj_noMM + -1*hcontam_noMM

#return 

NameError: name 'label_to_int_dict' is not defined

In [ ]:
Signals = {
    'ZPrime1' : ['1000', '1200', '1400', '1600', '1800', '2000', '2500', '3000', '3500', '4000', '4500'],
    'ZPrime10': ['1000', '1200', '1400', '1600', '1800', '2000', '2500', '3000', '3500', '4000', '4500', '5000'],
    'ZPrime30': ['1000', '1200', '1400', '1600', '1800', '2000', '2500', '3000', '3500', '4000', '4500', '5000'],
    'ZPrimeDM': ['1000', '1500', '2000', '2500', '3000', '3500', '4000', '4500', '5000'],
    'RSGluon':  ['1000', '1500', '2000', '2500', '3000', '3500', '4000', '4500', '5000']
}


cats = ['0bcen', '0bfwd', '1bcen', '1bfwd', '2bcen', '2bfwd']
cat_labels = ['cen0b', 'fwd0b', 'cen1b', 'fwd1b', 'cen2b', 'fwd2b']

systematics = [
        'jes',
        'jer',
        'pileup',
        'pdf',
        'q2',
        'btag',
        'toptagsf',
        'toptagxs',
        'lumi',
        'prefiring'
    ]

syst_labels = ['nominal']
for s in systematics:
    if not 'nominal' in s:
        syst_labels.append(s+'Down')
        syst_labels.append(s+'Up')
        

savefileheader = '../outputs/combine/TTbarAllHad{}_'.format(IOV.replace('20', '').replace('all',''))
# print(savefileheader)

froot = uproot.recreate(savefileheader+'CombineRoot_Cat_950_oldBTag_wideBins.root')

variable = 'ttbarmass'



for cat, catname in zip(cats, cat_labels):

    signal_cat = label_to_int_dict['2t'+cat]
    pretag_cat = label_to_int_dict['pret'+cat]

    hsignal = {}
    hsignal['ZPrime1'] = {}
    hsignal['ZPrime10'] = {}
    hsignal['ZPrime30'] = {}
    hsignal['ZPrimeDM'] = {}
    hsignal['RSGluon'] = {}
    
    
    HistDict = Use2016allIOV(variable, signal, cat)
    Hbkg = HistDict['ntmj'] + HistDict['ttbar']
    Hbkg_fixed = HistDict['ntmj_fixed'] + HistDict['ttbar']
    Ndenom = HistDict['antitag_data'].values() - HistDict['antitag_ttbar'].values()
    mistag_data = np.where(HistDict['pretag'].values()>0., HistDict['ntmj_fixed'].values() / HistDict['pretag'].values(), 0.)
    term1 = np.where(HistDict['pretag'].values()>0., 1. / HistDict['pretag'].values(), 0.)
    term2 = np.where( ((Ndenom*mistag_data)>0.), (np.ones(len(mistag_data))-mistag_data)/(Ndenom*mistag_data), 0. ) 
    mistagErrProp =  HistDict['ntmj_fixed']*np.sqrt( term1 + term2 )# 1 + 2 = mistag error & stat. error of NTMJ
    
    mmTerm1 = (HistDict['ntmj_fixed_noMM'].values() - HistDict['ntmj_fixed'].values()) # "source" of Mass Mod error
    mmTerm2 = (HistDict['ntmj_fixed_noMM'].values() + HistDict['ntmj_fixed'].values()) / 2.
    mmPercentErr = np.where( (mmTerm2>0.), mmTerm1/mmTerm2, 0. ) # Calculated percent error of the difference
    mmErrProp = HistDict['ntmj_fixed']*np.abs(mmPercentErr) # Mass Mod error
    
    qcdTerm1 = (HistDict['QCDbkgest'].values() - HistDict['QCDsignal'].values()) # "source" of QCD closure error
    qcdTerm2 = (HistDict['QCDbkgest'].values() + HistDict['QCDsignal'].values()) / 2.
    qcdPercentErr = np.where((qcdTerm2 > 0), qcdTerm1/qcdTerm2, 0.) # Calculated percent error of the difference
    qcdErrProp = HistDict['ntmj_fixed']*np.abs(qcdPercentErr) #QCD closure error

    froot["bkgest_"+catname+'_mistagUp'] = HistDict['ntmj_fixed'] + mistagErrProp
    froot["bkgest_"+catname+'_mistagDown'] = HistDict['ntmj_fixed'] + -1.0*mistagErrProp

    froot["bkgest_"+catname+'_massmodUp'] = HistDict['ntmj_fixed'] + mmErrProp
    froot["bkgest_"+catname+'_massmodDown'] = HistDict['ntmj_fixed'] + -1.0*mmErrProp

    froot["bkgest_"+catname+'_qcdclosureUp'] = HistDict['ntmj_fixed'] + qcdErrProp
    froot["bkgest_"+catname+'_qcdclosureDown'] = HistDict['ntmj_fixed'] + -1.0*qcdErrProp

    for syst in syst_labels:

        catsystString = catname+'_'+syst

        httbar_apv        = getHist(variable, 'TTbar', False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst})
        hcontam_apv       = getHist(variable, 'TTbar', True, '2016APV', sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':syst})
        httbar_noapv      = getHist(variable, 'TTbar', False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst})
        hcontam_noapv     = getHist(variable, 'TTbar', True, '2016', sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':syst})

        print('loading signals...')
        for sig in Signals.keys():
            for mass in Signals[sig]:
                hsignal_apv   = getHist(variable, sig, False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst}, masspoint=mass)
                hsignal_noapv = getHist(variable, sig, False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst}, masspoint=mass)
                hsignal[sig][mass] = hsignal_apv + hsignal_noapv

        httbar  = httbar_apv + httbar_noapv
        hcontam = hcontam_apv + hcontam_noapv

        print('filling root files...')
        if 'nominal' in syst:
            syst = ''
            catsystString = catname+syst
            hntmj_apv   = getHist(variable, 'JetHT', True, '2016APV',   sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
            hdata_apv   = getHist(variable, 'JetHT', False, '2016APV',  sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})
            hntmj_noapv = getHist(variable, 'JetHT', True, '2016',   sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
            hdata_noapv = getHist(variable, 'JetHT', False, '2016',  sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})

            hntmj = hntmj_apv + hntmj_noapv
            hdata = hdata_apv + hdata_noapv
            hntmj_fixed = hntmj + -1*hcontam
            print('data_obs_'+catsystString+'\nbkgest_'+catsystString)
            froot["data_obs_"+catsystString] = hdata
            froot["bkgest_"+catsystString] = hntmj_fixed

        print('TTbar_'+catsystString)
        froot["TTbar_"+catsystString] = httbar


        for sig in Signals.keys():

            if 'RSGluon' not in sig:
                if sig != 'ZPrime1': # ZPrime10, 30 and DM
                    [print(sig[:-2]+mass+'_'+sig[-2:]+'_'+catsystString) for mass in Signals[sig]]
                    for mass in Signals[sig]:
                        froot[sig[:-2]+mass+'_'+sig[-2:]+'_'+catsystString] = hsignal[sig][mass]
                else: # ZPrime1
                    [print(sig[:-1]+mass+'_'+sig[-1:]+'_'+catsystString) for mass in Signals[sig]]
                    for mass in Signals[sig]:
                        froot[sig[:-1]+mass+'_'+sig[-1:]+'_'+catsystString] = hsignal[sig][mass]
            else: # RSGluon
                [print(sig+mass+'_'+catsystString) for mass in Signals[sig]]
                for mass in Signals[sig]:
                    froot[sig+mass+'_'+catsystString] = hsignal[sig][mass]
                    
froot.close()            
            
            
            